# Stage 18 norm — FAIR decode (no oracle) on the saved checkpoint
The training eval picked min(CTC,attn) per clip USING the ground truth (oracle) ->
not comparable to the paper. This re-decodes the best-val checkpoint with fair
single-head decodes (CTC-only / attn-only), picks the better head ON VAL, and
reports TEST for that head. No re-training. GPU on (fast); ~2 min.

**Attach:** the landmark cache `gaurs86/wita-full-english-landmark-cache` AND your
checkpoint (upload `stage18_norm_best.pt` as a dataset, or attach the committed
run's output). Set `CKPT` in Cell 2.


## Cell 1 — clone + deps


In [ ]:
%%capture
!pip install editdistance scipy --quiet
import sys, os
!rm -rf /kaggle/working/wita_v2
!git clone -b stage13b-paper-replication "https://github.com/Gaurs86/WiTA-v2.git" '/kaggle/working/wita_v2'
sys.path.insert(0, '/kaggle/working'); sys.path.insert(0, '/kaggle/working/wita_v2')
for _m in [m for m in list(sys.modules) if m.split('.')[0] in ('wita_v2','stage16','stage17','stage18')]:
    del sys.modules[_m]
import torch


In [ ]:
print('GPU', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')


## Cell 2 — paths (EXPLICIT, no glob). Set CKPT to where you attached it.


In [ ]:
import os
CACHE_ROOT = '/kaggle/input/datasets/gaurs86/wita-full-english-landmark-cache/landmark_cache_122'
CKPT       = '/kaggle/input/datasets/gaurs86/stage18-norm-checkpoint/stage18_norm_best.pt'  # <-- EDIT to your path
assert os.path.isdir(os.path.join(CACHE_ROOT,'train')), f'fix CACHE_ROOT: {CACHE_ROOT}'
assert os.path.isfile(CKPT), f'fix CKPT: {CKPT}'
print('cache', CACHE_ROOT, '\nckpt ', CKPT)


## Cell 3 — normalization hook (MUST match training) + config


In [ ]:
import torch, random, numpy as np
import wita_v2.datasets.landmark_paper_split as lps
from stage18.normalize_cache import normalize_feature
if not getattr(lps.WiTAPaperSplitDataset, '_norm_patched', False):
    _orig = lps.WiTAPaperSplitDataset.__getitem__
    def _norm_getitem(self, i):
        out = _orig(self, i); x = out[0]
        return (torch.from_numpy(normalize_feature(x.detach().cpu().numpy())).to(x.dtype),) + tuple(out[1:])
    lps.WiTAPaperSplitDataset.__getitem__ = _norm_getitem
    lps.WiTAPaperSplitDataset._norm_patched = True
from wita_v2.configs.default import Config, DataConfig, EncoderConfig, TrainConfig
cfg = Config(data=DataConfig(hf_repo_id='yewon816/WiTA', lang='english', max_zips=None, max_frames=64, seed=42),
             encoder=EncoderConfig(arch='siglip'),
             train=TrainConfig(num_epochs=80, batch_size=32, lr=5e-4, weight_decay=5e-2, grad_clip=1.0,
                               num_workers=2, warmup_pct=0.05, seed=42, checkpoint_dir='/kaggle/working')).build()
print('normalization active | cfg device', cfg.device)


## Cell 4 — fair decode: CTC-only / attn-only, head chosen on VAL


In [ ]:
from torch.utils.data import DataLoader
from wita_v2.training.stage11_train import evaluate_loader, _collate
from wita_v2.models.conformer_ctc import ConformerCTC
from wita_v2.models.attention_decoder import AttentionDecoder
from wita_v2.datasets.landmark_paper_split import WiTAPaperSplitDataset
from wita_v2.datasets.vocab import make_converter
dev = cfg.device; conv = make_converter(cfg.data.lang)
pad, blank, sos, eos = cfg.vocab.pad_idx, cfg.vocab.blank_idx, cfg.vocab.sos_idx, cfg.vocab.eos_idx
enc = ConformerCTC(input_dim=190, vocab_size=cfg.vocab.ctc_vocab_size, d_model=256, n_layers=4, n_heads=4,
                   conv_kernel=15, dropout=0.2, upsample=2, input_layernorm=False).to(dev)
dec = AttentionDecoder(att_vocab_size=cfg.vocab.attn_vocab_size, bos_idx=sos, eos_idx=eos,
                       d_model=256, n_layers=2, n_heads=4, ff_mult=4, dropout=0.2).to(dev)
st = torch.load(CKPT, map_location='cpu', weights_only=False)
enc.load_state_dict(st['encoder_state_dict'], strict=False); dec.load_state_dict(st['decoder_state_dict'], strict=False)
print('loaded | ckpt best val_overall =', st.get('best_payload',{}).get('val_overall_cer','?'))

def fair(split):
    ds = WiTAPaperSplitDataset(CACHE_ROOT, split, subsets=('lex','nonlex'), converter=conv, transform=None)
    ld = DataLoader(ds, batch_size=32, shuffle=False, num_workers=2, collate_fn=lambda b:_collate(b,pad_idx=pad))
    pr = evaluate_loader(ld, encoder=enc, decoder=dec, cfg=cfg, blank=blank, sos=sos, eos=eos, pad=pad, device=dev)['pairs']
    cer = lambda rows,k: sum(r[k] for r in rows)/max(sum(r['L'] for r in rows),1)
    g = lambda k: {'overall':cer(pr,k), 'lex':cer([r for r in pr if r['subset']=='lex'],k),
                   'nonlex':cer([r for r in pr if r['subset']=='nonlex'],k)}
    return {'ctc':g('e_ctc'), 'attn':g('e_attn'), 'oracle':g('edit')}

val, test = fair('val'), fair('test')
head = 'ctc' if val['ctc']['overall'] <= val['attn']['overall'] else 'attn'
print(f"VAL  ctc={val['ctc']['overall']:.4f}  attn={val['attn']['overall']:.4f}  oracle={val['oracle']['overall']:.4f}")
print(f"--> fair head chosen on val: {head.upper()}")
t = test[head]
print(f"\n================ FAIR TEST ({head}) ================")
print(f"  overall {t['overall']:.4f}   lex {t['lex']:.4f}   nonlex {t['nonlex']:.4f}")
print(f"  paper   0.2924          0.2810       0.3650")
print("===================================================")
print("test ctc   :", {k:round(v,4) for k,v in test['ctc'].items()})
print("test attn  :", {k:round(v,4) for k,v in test['attn'].items()})
print("test oracle:", {k:round(v,4) for k,v in test['oracle'].items()}, '(NOT a valid claim)')
